# 2 · Email Triage Agent
## Structured Classification: From Free Text to Typed Results
⏱ ~15 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/2-email-triage/email_triage_workbook.ipynb)

Every operations team drowns in email. Prioritizing what needs action *right now* versus what can wait is a full-time job — and it's exactly the kind of judgment call that a well-prompted LLM can automate.

This example builds an agent that reads a raw email and returns a **typed classification**: urgency, category, a one-line summary, and a recommended action. No free text output — a Pydantic schema enforces the shape of every response.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | The business problem: email overload |
| 2 | Why free text output fails in production |
| 3 | Designing a Pydantic schema |
| 4 | `with_structured_output()` — binding a schema to a model |
| 5 | Running the triage agent |
| ★ | Exercise: extend the schema, test new emails |

---

### Prerequisites
- Python 3.10+
- `OPENAI_API_KEY` in `.env` or Colab Secrets

### Key References
> OpenAI Structured Outputs: https://platform.openai.com/docs/guides/structured-outputs  
> Pydantic docs: https://docs.pydantic.dev  
> LangChain structured output: https://python.langchain.com/docs/concepts/structured_outputs/

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "pydantic", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local environment — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Loaded API key from Colab Secrets.")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    print("Loaded API key from .env file.")

key = os.environ.get("OPENAI_API_KEY", "")
if key and key.startswith("sk-"):
    print("✓ OPENAI_API_KEY is set and looks valid.")
else:
    print("✗ OPENAI_API_KEY missing or invalid.")
    print("  Colab: Secrets panel (key icon) → Add secret 'OPENAI_API_KEY'")
    print("  Local: create a .env file with OPENAI_API_KEY=sk-...")

---

## Part 1 — The Business Problem

---

An operations manager at a mid-size company receives 150+ emails a day across:
- Overdue invoices and payment disputes
- IT system outages and access requests  
- General HR inquiries and scheduling
- Vendor spam

Without a system, *every* email looks equally urgent in an inbox. A triage agent solves this by classifying each email the moment it arrives:

```
incoming email
       ↓
  triage agent
       ↓
urgency: high        ← this one needs attention NOW
category: billing
summary: Invoice #4821 overdue 30 days, account suspended
action: escalate to finance team and AP manager
```

The ops manager now sees a prioritized queue instead of a pile.

---

## Part 2 — Why Free Text Output Fails

---

The naive approach: ask the LLM to classify the email in plain text.

```python
response = llm.invoke(f"Classify this email: {email}")
# response.content might be:
# "This email is very urgent and seems to be about a billing issue..."
# "URGENT - Billing"  
# "{"urgency": "HIGH", "type": "billing"}"
# "The urgency level is high (billing dispute)"
```

Every response has a different format. Your code needs to parse four different structures. On the next model update, the format changes again.

**The fix:** constrain the model to return a Pydantic schema. The output is *always* the same shape — validated, typed, and ready to use.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

SAMPLE_EMAIL = """
Subject: URGENT - Invoice #4821 overdue 30 days, account suspended
From: accounts@acme-corp.com

Hi team, our account has been suspended due to unpaid invoice #4821 ($4,200).
Payment was due May 1st -- now 30 days overdue. We need this resolved 
immediately or we lose access to production systems. Please escalate ASAP.
"""

# Without structured output — unpredictable format
raw = llm.invoke(f"Classify this email by urgency and category:\n{SAMPLE_EMAIL}")
print("Free text response:")
print(repr(raw.content))
print()
print("Problem: how do you reliably parse this into urgency + category?")

---

## Part 3 — Designing the Schema

---

A Pydantic schema defines exactly what fields the model must populate. `Literal[...]` values constrain the model to a fixed set of options — it cannot return `"very urgent"` or `"kinda important"`, only `"high"`, `"medium"`, or `"low"`.

**Schema design checklist:**
- Every field has a `Field(description=...)` — this is what the model reads to know what to fill in
- Constrained choices use `Literal[...]` — enforced at the API level, not just the prompt
- Text fields specify expected length in the description

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
import json


class EmailTriage(BaseModel):
    urgency: Literal["high", "medium", "low"] = Field(
        description="Urgency level: 'high' if immediate action required, "
                    "'medium' if action needed within 24h, 'low' if informational"
    )
    category: Literal["billing", "technical", "hr", "general", "spam"] = Field(
        description="Email category based on subject matter"
    )
    summary: str = Field(
        description="One sentence (max 20 words) summarizing what the email is about"
    )
    recommended_action: str = Field(
        description="One concrete action the recipient should take next"
    )


# This JSON Schema is what gets sent to the OpenAI API
print("Schema sent to the model:")
print(json.dumps(EmailTriage.model_json_schema(), indent=2))

---

## Part 4 — `with_structured_output()`: Binding the Schema

---

`llm.with_structured_output(EmailTriage)` returns a runnable that:
1. Sends the schema to the API via function calling (not in the prompt)
2. Constrains the model to emit only valid schema tokens
3. Parses the response through Pydantic
4. Returns a typed `EmailTriage` instance — never a string

```
email text  →  LLM + EmailTriage schema  →  validated EmailTriage object
                                             (the model cannot deviate from the schema)
```

In [ ]:
# Bind the schema to the model
classifier = llm.with_structured_output(EmailTriage)

result = classifier.invoke(SAMPLE_EMAIL)

print(f"Return type: {type(result).__name__}")
print()
print(f"Urgency:  {result.urgency}")
print(f"Category: {result.category}")
print(f"Summary:  {result.summary}")
print(f"Action:   {result.recommended_action}")

---

## Part 5 — Testing Multiple Email Types

---

Let's run the same classifier on different emails to see how it handles varied inputs.

In [ ]:
EMAILS = [
    ("Overdue invoice", """Subject: URGENT - Invoice #4821 overdue 30 days, account suspended
Hi team, our account has been suspended due to unpaid invoice #4821 ($4,200).
Payment was due May 1st -- 30 days overdue. Please escalate ASAP."""),

    ("Server outage", """Subject: ALERT: Production database unreachable - all services down
Our monitoring system has detected that db-prod-01 is not responding.
All customer-facing services are currently returning 503 errors. 
Engineering team has been paged but we need help from ops."""),

    ("Team lunch", """Subject: Team lunch Thursday?
Hey everyone, just wanted to see if people are free for lunch this Thursday
around noon? There's a new Thai place that just opened nearby. Let me know!"""),
]

print(f"{'EMAIL':<20} {'URGENCY':<10} {'CATEGORY':<12} {'SUMMARY'}")
print("-" * 80)
for label, email in EMAILS:
    r = classifier.invoke(email)
    print(f"{label:<20} {r.urgency:<10} {r.category:<12} {r.summary[:40]}")

---

## Exercise — Extend the Schema and Test Your Own Emails

**Part A:** Add a `requires_reply` field to `EmailTriage` — a boolean that indicates whether the email needs a reply (not just action).

**Part B:** Test the extended classifier on an email you write yourself.

**Questions to consider:**
1. Does the model correctly set `requires_reply=True` for the overdue invoice?
2. Does it set `requires_reply=False` for a system alert?
3. What happens with ambiguous emails?

In [ ]:
# ── Exercise Starter ────────────────────────────────────────────────────────
from pydantic import BaseModel, Field
from typing import Literal

class EmailTriageV2(BaseModel):
    urgency: Literal["high", "medium", "low"] = Field(description="...")
    category: Literal["billing", "technical", "hr", "general", "spam"] = Field(description="...")
    summary: str = Field(description="...")
    recommended_action: str = Field(description="...")
    # TODO: add requires_reply: bool with a good description
    # requires_reply: bool = Field(description="...")

# TODO: bind to classifier_v2 = llm.with_structured_output(EmailTriageV2)
# TODO: test on the emails above
# TODO: write your own test email and classify it

In [ ]:
# ── Answer Key ──────────────────────────────────────────────────────────────

class EmailTriageV2Answer(BaseModel):
    urgency: Literal["high", "medium", "low"] = Field(
        description="Urgency level: 'high' if immediate action required, "
                    "'medium' if action needed within 24h, 'low' if informational"
    )
    category: Literal["billing", "technical", "hr", "general", "spam"] = Field(
        description="Email category based on subject matter"
    )
    summary: str = Field(description="One sentence (max 20 words) summarizing what the email is about")
    recommended_action: str = Field(description="One concrete action the recipient should take next")
    requires_reply: bool = Field(
        description="True if the sender is explicitly waiting for a response or decision, "
                    "False if it is a notification, alert, or FYI"
    )

classifier_v2 = llm.with_structured_output(EmailTriageV2Answer)

for label, email in EMAILS:
    r = classifier_v2.invoke(email)
    print(f"{label:<20} urgency={r.urgency:<8} reply={str(r.requires_reply):<6} {r.summary[:40]}")

---

## What's Next?

You now understand how to constrain LLM output to a typed schema — the single highest-leverage pattern in applied LLM engineering.

- **Example 3 — Invoice Extractor** ([`../3-invoice-extractor/`](../3-invoice-extractor/)): take structured output further with nested schemas (`Invoice` containing a list of `LineItem` objects) and field validators that cause automatic retries.
- **Example 4 — Lead Qualifier** ([`../4-lead-qualifier/`](../4-lead-qualifier/)): combine structured output with a rubric in the system prompt so the model must explain *why* it gave a score.

### Further Reading
- OpenAI Structured Outputs: https://platform.openai.com/docs/guides/structured-outputs
- Pydantic `Literal` types: https://docs.pydantic.dev/latest/concepts/types/
- LangChain `with_structured_output`: https://python.langchain.com/docs/concepts/structured_outputs/

---
*Workshop complete. Next: Example 3 — Invoice Extractor.*